[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/berdakh/ROBT407-SVM-Labs/blob/main/student/204_soft_margin_svm.ipynb)

# ROBT 407 · Soft-margin SVMs: slack, bounded multipliers, and validation

**Student workbook | Python 3 | Core: 90 minutes | Deepening work + mini-project: 2–3 hours**

Implement hinge loss and the primal objective; distinguish free and bounded support vectors; explore C and gamma; tune a scaling pipeline without test leakage.

Use the cycle **predict → run → inspect → explain**. Write predictions before running experiments. Complete the TODO functions and rerun their checks. Synthetic experiments and bundled scikit-learn datasets are available locally; no dataset downloads are needed. Core activities follow the lecture; deepening labs and mini-projects extend it.

## Lecture alignment and course context
Source: **204_handout (1).pdf**. PDF pages 3–7: noise, violations and primal; 8–11: dual bounds; 12–16: free and bounded support vectors; 17–19: model selection and cross-validation. Page numbers refer to physical PDF pages, starting at 1, rather than the slide footer.

Course context: ROBT 407, Fall 2026; these activities support theoretical reasoning (CLO1), pipeline design and evaluation (CLO2), and Python implementation (CLO3). Textbook context: *Learning From Data*, Yaser S. Abu-Mostafa, Malik Magdon-Ismail, Hsuan-Tien Lin. No textbook excerpt was supplied; these are original explanations and exercises based on the uploaded slides, with no claimed textbook page mapping.

The lecture-specific quiz checks understanding of slide examples; it does not verify whether someone watched a video.

**Notation:** rows are examples; labels are −1 and +1 unless stated otherwise. Bias is separate from w. Margin means one-sided geometric margin unless explicitly called full width.

## Setup
Run cells in order in JupyterLab, Jupyter Notebook, or Google Colab with a Python 3 kernel. Required packages: NumPy, SciPy, Matplotlib, scikit-learn; ipywidgets is optional. Static plots and editable function calls remain available without widgets. Live sliders are opt-in: set ENABLE_WIDGETS=True in Setup and rerun an experiment cell. Static figures are the default for predictable batch execution. Student notebooks intentionally leave implementations incomplete; their checks print reminders until completed.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from sklearn.svm import SVC
from sklearn.datasets import make_moons, make_circles
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, confusion_matrix
from IPython.display import display
np.set_printoptions(precision=5, suppress=True)
plt.rcParams.update({'figure.figsize': (7, 5), 'axes.grid': True, 'font.size': 11})
SEED = 407

def boundary(model, X, y, title='', ax=None):
    if ax is None: _, ax = plt.subplots()
    low, high = X.min(axis=0)-.6, X.max(axis=0)+.6
    xx, yy = np.meshgrid(np.linspace(low[0],high[0],160), np.linspace(low[1],high[1],160))
    zz = model.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, zz, levels=[-np.inf,0,np.inf], colors=['#e7edff','#ffe9da'], alpha=.8)
    ax.contour(xx, yy, zz, levels=[-1,0,1], colors='black', linestyles=['--','-','--'])
    ax.scatter(X[:,0],X[:,1],c=y,cmap='coolwarm',edgecolors='black',s=35)
    if hasattr(model,'support_'):
        sv = X[model.support_]
        ax.scatter(sv[:,0],sv[:,1],s=115,facecolors='none',edgecolors='#208050',linewidths=1.5,label='Support vectors')
        ax.legend(loc='best')
    ax.set(xlabel='$x_1$',ylabel='$x_2$',title=title)
    return ax

ENABLE_WIDGETS = False  # Set True and rerun the experiment cells for live sliders.

def optional_slider(fn, name, lo, hi, value, step):
    if not ENABLE_WIDGETS:
        print("To enable live sliders, set ENABLE_WIDGETS = True in Setup; direct function calls also work.")
        return
    # Static experiments remain available if this frontend cannot display widgets.
    try:
        import ipywidgets as widgets
        display(widgets.interactive(fn, **{name: widgets.FloatSlider(min=lo,max=hi,value=value,step=step,continuous_update=False)}))
    except ImportError:
        print('Widgets unavailable; call the experiment function with a new value.')

## 1. Replace perfect separation with controlled violations (10 min)
The soft-margin primal permits slack $\xi_n\ge0$:
$$\min_{w,b,\xi}\tfrac12\|w\|^2+C\sum_n\xi_n,
\quad y_n(w^Tx_n+b)\ge1-\xi_n.$$
For fixed $w,b$ and $C>0$, the smallest feasible slack is
$$\xi_n=\max(0,1-y_nf(x_n)).$$
Thus the equivalent unconstrained objective uses hinge loss:
$$J(w,b)=\tfrac12\|w\|^2+C\sum_n\max(0,1-y_nf(x_n)).$$
$C$ controls the penalty per unit violation in this **sum-loss convention**. Larger $C$ penalizes violations more strongly. Compare objectives only when their normalization matches.

If $0<y_nf(x_n)<1$, the prediction is correct but has a margin violation. If $y_nf(x_n)<0$, it is misclassified and has slack greater than 1. A score of zero is a boundary case.

### Implement A: hinge loss and objective (10 min)
Return the per-example hinge losses and then implement the objective using a sum, not a mean.

In [ ]:
def hinge(y,scores):
    # TODO: one loss per example.
    return None

def primal_objective(w,b,X,y,C):
    # TODO: scalar objective.
    return None

In [ ]:
loss=hinge(np.ones(4),np.array([1.5,1.,.3,-.2]))
if loss is None: print('Complete Implement A, then rerun.')
else:
    np.testing.assert_allclose(loss,[0,0,.7,1.2])
    test_X=np.array([[1.,0.],[0.,1.]])
    np.testing.assert_allclose(primal_objective(np.array([1.,1.]),0,test_X,np.array([1.,-1.]),2),5.)
    print('Hinge loss and objective checks passed.')

**Your reasoning:** write your prediction, observations and explanation here.

## 2. The dual changes by one upper bound (10 min)
Stationarity in slack gives $C-\alpha_n-\beta_n=0$, where both multipliers are nonnegative. Therefore
$$\max_\alpha\ \mathbf1^T\alpha-\tfrac12\alpha^TQ\alpha,
\quad y^T\alpha=0,\quad 0\le\alpha_n\le C.$$
Complementary slackness gives
$$\alpha_n(1-\xi_n-y_nf(x_n))=0,\qquad (C-\alpha_n)\xi_n=0.$$
- $\alpha_n=0$: outside or on a margin plane, with zero slack.
- $0<\alpha_n<C$: **free** support vector, exactly on a margin plane, zero slack.
- $\alpha_n=C$: **bounded** support vector; it can be on or inside a margin plane, including misclassified points. Bounded does not mean necessarily misclassified.

Compute $b=y_s-\sum_n\alpha_ny_nK(x_n,x_s)$ from a **free** support vector. If no free vector exists, KKT gives an interval for $b$ rather than this direct formula.

In [ ]:
X,y01=make_moons(n_samples=150,noise=.28,random_state=SEED)
y=2*y01-1
C_value=1.
linear=SVC(kernel='linear',C=C_value,tol=1e-9).fit(X,y)
alpha=np.zeros(len(y));alpha[linear.support_]=np.abs(linear.dual_coef_[0])
scores=linear.decision_function(X);margins=y*scores
slack=np.maximum(0,1-margins)
free=(alpha>1e-6)&(alpha<C_value-1e-6)
bounded=alpha>=C_value-1e-6
print('free, bounded, misclassified:',free.sum(),bounded.sum(),(margins<0).sum())
print('correctly classified but violating margin:',((margins>0)&(margins<1)).sum())
print('first 12 rows: alpha, margin, slack')
print(np.c_[alpha,margins,slack][:12])
print('max KKT residuals:',np.max(np.abs(alpha*(1-slack-margins))),np.max(np.abs((C_value-alpha)*slack)))
assert np.max(np.abs(alpha*(1-slack-margins)))<1e-4
boundary(linear,X,y,'Soft-margin linear SVM');plt.show()

### Implement B: recover b from free support vectors (10 min)
Given a training Gram matrix and the unsigned multipliers, return the mean bias over free support vectors. Return None when no free vector is available.

In [ ]:
def bias_from_free(K,y,alpha,C,tol=1e-6):
    # TODO: select 0 < alpha < C and average the corresponding bias estimates.
    return None

In [ ]:
estimate=bias_from_free(X@X.T,y,alpha,C_value)
if estimate is None: print('No estimate: finish Implement B, or inspect whether free vectors exist.')
else:
    np.testing.assert_allclose(estimate,linear.intercept_[0],atol=2e-4)
    assert bias_from_free(np.eye(2),np.array([-1.,1.]),np.ones(2),1.) is None
    print('Free-vector bias checks passed.')

**Your reasoning:** write your prediction, observations and explanation here.

## 3. Explore C and gamma together (10 min)
Predict the effect of each parameter before looking at the plots. Explain the boundary changes and count support vectors. These plots are exploratory; do not use the final test set to choose a pleasing boundary.

In [ ]:
fig,axs=plt.subplots(2,3,figsize=(14,8))
for i,Cv in enumerate([.1,100]):
    for j,gamma in enumerate([.1,1,30]):
        m=SVC(C=Cv,gamma=gamma).fit(X,y)
        boundary(m,X,y,f'C={Cv}, gamma={gamma}, SV={len(m.support_)}',axs[i,j])
plt.tight_layout();plt.show()
def penalty_experiment(log_C=0.):
    m=SVC(C=10**log_C,gamma=1.).fit(X,y)
    boundary(m,X,y,f'C={10**log_C:.3g}, gamma=1');plt.show()
optional_slider(penalty_experiment,'log_C',-2.,2.,0.,.25)

## 4. Choose a model without leaking information (15 min)
Reserve a stratified test set first. On the remaining development data, run a stratified cross-validation grid. Put scaling **inside** the pipeline so each fold learns its mean and variance only from its training partition.
$$E_{CV}(C,\gamma)=\frac1V\sum_{v=1}^V E_{val}^{(v)}(C,\gamma).$$
Choose using development CV only, refit on all development data, then evaluate the test set once. CV is used for model selection, so the best CV score can be optimistic; the held-out test gives a separate estimate. Do not change the grid after inspecting that test score.

In [ ]:
Xall,yall=make_moons(n_samples=360,noise=.25,random_state=SEED+2)
Xdev,Xtest,ydev,ytest=train_test_split(Xall,yall,test_size=.25,stratify=yall,random_state=SEED)
pipe=Pipeline([('scale',StandardScaler()),('svc',SVC())])
grid={'svc__C':[.1,1,10,100],'svc__gamma':[.03,.3,3,30]}
cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=SEED)
search=GridSearchCV(pipe,grid,cv=cv,scoring='accuracy',n_jobs=1,return_train_score=True)
search.fit(Xdev,ydev)
print('CV-selected parameters:',search.best_params_)
print('best mean CV accuracy:',search.best_score_)
means=search.cv_results_['mean_test_score'].reshape(4,4)
fig,ax=plt.subplots();im=ax.imshow(means,origin='lower',vmin=.5,vmax=1,cmap='viridis')
ax.set(xticks=range(4),xticklabels=grid['svc__gamma'],yticks=range(4),yticklabels=grid['svc__C'],xlabel='gamma',ylabel='C',title='Development-only cross-validation accuracy')
for i in range(4):
    for j in range(4):ax.text(j,i,f'{means[i,j]:.3f}',ha='center',va='center',color='white' if means[i,j]<.8 else 'black')
fig.colorbar(im,ax=ax);plt.show()
# Final evaluation: do not tune against this output.
pred=search.predict(Xtest)
print('held-out test accuracy:',accuracy_score(ytest,pred))
print('test confusion matrix (rows=true, columns=predicted):'); print(confusion_matrix(ytest,pred))

## 5. Explain and submit (10 min)
1. Find one correctly classified margin violator and one misclassified point. Report their scores, labels and slack.
2. Explain why the bounded-support-vector count is an upper bound on strict training misclassifications, not the number of misclassifications.
3. Why is scaling before cross-validation a leakage problem even though labels are not used by the scaler?
4. Propose an engineering classification application. Which data split would be appropriate if observations come from repeated measurements of the same robot? Explain why a random row split may overestimate generalization.

**Submit:** both implementations, the six-panel parameter comparison, the CV table, the final test result and your explanation of the evaluation protocol. The robot scenario is an extension: grouped splitting would be needed to evaluate generalization to unseen robots.

## Deepening lab A · Hinge loss versus 0/1 error (15 min)
For signed score $r=yf(x)$, compare $\ell_{01}(r)=\mathbf1[r\le0]$ (our tie convention) with $\ell_h(r)=\max(0,1-r)$. The hinge upper-bounds 0/1 error under this convention. Minimizing hinge loss is a convex surrogate problem, not direct minimization of the error count.

Away from $r_n=1$, the sum-loss objective has a subgradient
$$\partial_w J=w-C\sum_{n:r_n<1}y_nx_n,\qquad
\partial_b J=-C\sum_{n:r_n<1}y_n.$$
At the hinge, choosing zero contribution is an admissible subgradient. Explain why neither the bias nor its derivative has an L2-penalty term.

In [ ]:
r=np.linspace(-2,3,400)
fig,ax=plt.subplots();ax.plot(r,(r<=0).astype(float),label='0/1 loss (ties counted as errors)')
ax.plot(r,np.maximum(0,1-r),label='Hinge loss');ax.axvline(0,color='gray',linestyle=':');ax.axvline(1,color='gray',linestyle=':')
ax.set(xlabel='Signed score y f(x)',ylabel='Loss',title='Correct predictions may still incur hinge loss');ax.legend();plt.show()
fig,ax=plt.subplots()
category=np.where(alpha<1e-6,0,np.where(alpha<C_value-1e-6,1,2))
for label,marker,name in [(0,'o','Non-support'),(1,'s','Free support'),(2,'^','Bounded support')]:
    selected=category==label
    ax.scatter(margins[selected],slack[selected],marker=marker,label=name,alpha=.7)
ax.axvline(0,color='gray',linestyle=':');ax.axvline(1,color='gray',linestyle=':')
ax.set(xlabel='Functional margin',ylabel='Minimum slack',title='KKT categories along the hinge');ax.legend();plt.show()

### Implement C · A linear SVM subgradient
Return the weight and bias subgradients of the sum-loss objective. Verify with central differences at points away from the hinge. Explain why finite differences need special care at a nondifferentiable point.

In [ ]:
def svm_subgradient(w,b,X,y,C):
    # TODO: use examples with functional margin < 1.
    return None

In [ ]:
wt=np.array([.37,-.61]);bt=.23;Ct=.4
sg=svm_subgradient(wt,bt,X,y,Ct)
if sg is None: print('Complete Implement C (and hinge Implement A).')
else:
    u=np.r_[wt,bt];eye=np.eye(3);eps=1e-6
    obj=lambda z:primal_objective(z[:2],z[2],X,y,Ct)
    numeric=np.array([(obj(u+eps*e)-obj(u-eps*e))/(2*eps) for e in eye])
    np.testing.assert_allclose(np.r_[sg[0],sg[1]],numeric,atol=1e-6)
    print('Subgradient finite-difference check passed away from hinge points.')

**Your reasoning:** write your prediction, observations and explanation here.

### Implement D · Soft-margin dual QP
Given a Gram matrix, labels and C, solve the dual with box bounds and the equality constraint. Return unsigned alpha. Use SciPy minimize with SLSQP and an analytic objective gradient; check its success flag.

In [ ]:
def soft_dual(K,y,C):
    # TODO: minimize .5*a@Q@a - a.sum(), with y@a=0 and 0<=a<=C.
    return None

In [ ]:
smallX=np.array([[-2.,0.],[-1.,.2],[-.2,0.],[.2,0.],[1.,-.2],[2.,0.]])
small_y=np.array([-1.,-1.,1.,-1.,1.,1.]);smallC=.8
smallK=smallX@smallX.T
small_alpha=soft_dual(smallK,small_y,smallC)
if small_alpha is None: print('Complete Implement D (and B for bias recovery).')
else:
    assert small_alpha.min()>=-1e-7 and small_alpha.max()<=smallC+1e-7
    assert abs(small_y@small_alpha)<1e-7
    small_b=bias_from_free(smallK,small_y,small_alpha,smallC)
    assert small_b is not None
    small_w=(small_alpha*small_y)@smallX
    ref=SVC(kernel='linear',C=smallC,tol=1e-9).fit(smallX,small_y)
    np.testing.assert_allclose(smallX@small_w+small_b,ref.decision_function(smallX),atol=1e-4)
    primal=primal_objective(small_w,small_b,smallX,small_y,smallC)
    dual=small_alpha.sum()-.5*small_w@small_w
    assert abs(primal-dual)<1e-5
    print('Box constraints, prediction agreement and primal-dual gap passed:',primal-dual)

**Your reasoning:** write your prediction, observations and explanation here.

### Implement E · Count the model-selection workload
Accept a dictionary grid or a list of conditional dictionaries. Return (candidate count, fold-fit count, total fits including one refit). Empty dictionaries represent one default candidate. Do not multiply irrelevant kernel parameters into the linear branch.

In [ ]:
def search_workload(grids,folds=5):
    # TODO: count Cartesian products within branches; add across branches.
    return None

In [ ]:
conditional=[{'svc__kernel':['linear'],'svc__C':[.1,1,10]},
 {'svc__kernel':['rbf'],'svc__C':[.1,1,10],'svc__gamma':[.01,.1,1]}]
work=search_workload(conditional,5)
if work is None: print('Complete Implement E.')
else:
    assert work==(12,60,61)
    assert search_workload({},5)==(1,5,6)
    print('Candidates, fold fits, fits including refit:',work)

**Your reasoning:** write your prediction, observations and explanation here.

## Real-data mini-project · Breast Cancer Wisconsin classification study (45–75 min)
569 observations, 30 numerical measurements and two supplied classes. This historical dataset teaches classification evaluation; these experiments do not establish clinical usefulness.

**Before fitting:** write a hypothesis about the useful kernel or regularization strength, the expected errors and the effect of scaling.

**Search budget:** Tune two numerical hyperparameters, C and gamma. Four values each give 16 candidates, 80 five-fold fits and one refit. The RBF kernel family is fixed.

Use a stratified 80/20 development/test split. Hyperparameter selection uses **five-fold stratified cross-validation** on development data only. Fit scaling inside each fold through a pipeline. Reserve test labels until selection and refitting are finished. These datasets are bundled with scikit-learn; the loaders need no dataset download. Small benchmark results demonstrate a workflow rather than establish deployment performance.

### Understand the model-selection choices
| Choice | Effect | Applicable kernels |
|---|---|---|
| $C>0$ | Penalty per unit margin violation in the sum-loss objective | Linear, polynomial, RBF |
| $\gamma>0$ | Input scale in similarities | Polynomial, RBF |
| degree | Polynomial power | Polynomial |
| coef0 | Additive polynomial constant | Polynomial |
| kernel | Similarity family and feature geometry | Family comparison |

Distinguish hyperparameter **names**, candidate **values** per name, model **combinations** and total fold **fits**. Conditional grids add branch workloads instead of multiplying irrelevant parameters into every family. Five CV training/validation rotations are performed for each candidate; the winner is refitted on all development data.

SVC handles multiclass tasks using pairwise classifiers. Its packed multiclass coefficient array is not the single binary alpha vector used in earlier derivations.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import ParameterGrid, cross_val_score
from sklearn.dummy import DummyClassifier
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, balanced_accuracy_score
import pandas as pd
project_data=load_breast_cancer()
project_X=project_data.data.copy();project_y=project_data.target.copy()
project_names=np.asarray(project_data.target_names).astype(str)

X_develop,X_holdout,y_develop,y_holdout=train_test_split(project_X,project_y,test_size=.2,stratify=project_y,random_state=SEED+10)
project_cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=SEED+11)
project_grid={'svc__kernel':['rbf'],'svc__C':[.01,.1,1,10],'svc__gamma':[.001,.01,.1,1]}
project_candidates=len(list(ParameterGrid(project_grid)))
print('Data shape and class counts:',project_X.shape,np.unique(project_y,return_counts=True))
print('Development/test sizes:',len(y_develop),len(y_holdout))
print('Candidates, fold fits, fits including refit:',project_candidates,5*project_candidates,5*project_candidates+1)
assert project_cv.n_splits==5
visits=np.zeros(len(y_develop),dtype=int)
for training,validation in project_cv.split(X_develop,y_develop):
    assert len(np.intersect1d(training,validation))==0
    visits[validation]+=1
assert np.all(visits==1)

### Visual audit before fitting
Inspect development data and class counts. Any exploratory scaling below is separate from the estimator; the CV pipeline still learns its own scaler separately within every fold. For digit images, inspect pixels rather than just flattened vectors.

In [ ]:
fig,axs=plt.subplots(1,2,figsize=(12,4))
labels,counts=np.unique(y_develop,return_counts=True)
axs[0].bar(project_names[labels],counts);axs[0].set(title='Development class counts',ylabel='Observations')
exploreZ=StandardScaler().fit_transform(X_develop)
for label in labels:
    axs[1].scatter(exploreZ[y_develop==label,0],exploreZ[y_develop==label,1],label=project_names[label],alpha=.65)
axs[1].set(xlabel=str(project_data.feature_names[0])+' (scaled)',ylabel=str(project_data.feature_names[1])+' (scaled)',title='Two-feature view; classifier uses all features')
axs[1].legend();plt.tight_layout();plt.show()

### Implement F · Construct a leakage-safe five-fold search
Return an **unfitted** GridSearchCV with a StandardScaler/SVC pipeline named scale and svc. Use the supplied splitter, accuracy scoring, refit=True, return_train_score=True, n_jobs=1 and error_score='raise'.

In [ ]:
def make_project_search(grid,splitter):
    # TODO: construct the pipeline and search.
    return None

In [ ]:
candidate_search=make_project_search(project_grid,project_cv)
if candidate_search is None: print('Complete Implement F to construct your own search.')
else:
    assert isinstance(candidate_search,GridSearchCV)
    assert isinstance(candidate_search.estimator,Pipeline)
    assert isinstance(candidate_search.estimator.named_steps['scale'],StandardScaler)
    assert isinstance(candidate_search.estimator.named_steps['svc'],SVC)
    assert candidate_search.cv.n_splits==5 and candidate_search.scoring=='accuracy'
    assert candidate_search.refit and candidate_search.return_train_score
    assert candidate_search.n_jobs==1 and candidate_search.error_score=='raise'
    assert not hasattr(candidate_search,'cv_results_')
    print('Search configuration checks passed.')

**Your reasoning:** write your prediction, observations and explanation here.

### Run the reference workflow, then compare your implementation
This reference is complete so the project can be explored before finishing Implement F. To test your own implementation, replace the reference construction with your function's returned search. The reference supplies the protocol, not your written analysis or conclusions. All tuning uses development data only.

In [ ]:
project_pipeline=Pipeline([('scale',StandardScaler()),('svc',SVC(cache_size=200))])
project_search=GridSearchCV(project_pipeline,project_grid,cv=project_cv,scoring='accuracy',
    refit=True,return_train_score=True,n_jobs=1,error_score='raise')
project_search.fit(X_develop,y_develop)
project_results=pd.DataFrame(project_search.cv_results_)
parameter_columns=[column for column in project_results if column.startswith('param_')]
summary_columns=['rank_test_score',*parameter_columns,'mean_train_score','mean_test_score','std_test_score','mean_fit_time']
display(project_results[summary_columns].sort_values('rank_test_score').head(12))
print('Selected parameters:',project_search.best_params_)
print('Selection mean CV accuracy:',project_search.best_score_)
print('Fold spread is descriptive, not an independent-sample confidence interval.')
baseline_scores=cross_val_score(DummyClassifier(strategy='most_frequent'),X_develop,y_develop,cv=project_cv,scoring='accuracy')
print('Majority baseline mean CV accuracy:',baseline_scores.mean())

In [ ]:
top=project_results.sort_values('rank_test_score').head(10).reset_index(drop=True)
fig,axs=plt.subplots(1,2,figsize=(14,4));x=np.arange(len(top))
axs[0].errorbar(x,top['mean_test_score'],yerr=top['std_test_score'],fmt='o',capsize=4,label='Validation mean +/- fold SD')
axs[0].plot(x,top['mean_train_score'],'s--',label='Mean training accuracy')
axs[0].axhline(baseline_scores.mean(),color='gray',linestyle=':',label='Majority baseline')
axs[0].set(xlabel='Candidate rank (top ten)',ylabel='Accuracy',title='Selection evidence and generalization gap');axs[0].legend(fontsize=8)
axs[1].scatter(project_results['mean_fit_time'],project_results['mean_test_score'],c=project_results['mean_train_score'],cmap='viridis',edgecolors='black')
axs[1].set(xlabel='Mean fit time per fold (seconds)',ylabel='Mean validation accuracy',title='Performance versus fitting cost')
plt.tight_layout();plt.show()

In [ ]:
rbf_results=project_results[project_results['param_svc__kernel']=='rbf']
heat=rbf_results.pivot(index='param_svc__C',columns='param_svc__gamma',values='mean_test_score').sort_index().sort_index(axis=1)
fig,ax=plt.subplots();im=ax.imshow(heat.values,origin='lower',cmap='viridis')
ax.set(xticks=np.arange(len(heat.columns)),xticklabels=heat.columns,yticks=np.arange(len(heat.index)),yticklabels=heat.index,xlabel='gamma',ylabel='C',title='RBF: five-fold development accuracy')
for i in range(len(heat.index)):
    for j in range(len(heat.columns)):ax.text(j,i,f'{heat.iloc[i,j]:.3f}',ha='center',va='center',color='white' if heat.iloc[i,j]<np.median(heat.values) else 'black')
fig.colorbar(im,ax=ax);plt.show()

### Learning curve before final testing
Use the selected hyperparameters in a five-fold development learning curve. Scaling stays inside each training subset. This descriptive curve is computed after model selection; it is not a fresh unbiased evaluation of the winner. Compare gaps as training size grows.

In [ ]:
from sklearn.model_selection import learning_curve
sizes,training_scores,validation_scores=learning_curve(project_search.best_estimator_,X_develop,y_develop,
    cv=project_cv,train_sizes=[.3,.6,1.],scoring='accuracy',n_jobs=1,shuffle=True,random_state=SEED)
fig,ax=plt.subplots()
for scores,name in [(training_scores,'Training'),(validation_scores,'Validation')]:
    mean=scores.mean(axis=1);sd=scores.std(axis=1)
    ax.plot(sizes,mean,'o-',label=name);ax.fill_between(sizes,mean-sd,mean+sd,alpha=.15)
ax.set(xlabel='Training examples per fold',ylabel='Accuracy',title='Development learning curve (selected parameters)');ax.legend();plt.show()

### Freeze the choice, then evaluate once on the held-out test set
Write down the selected configuration and expected failure modes before running this cell. Do not alter the grid based on these results. A new hypothesis after seeing test results requires a fresh evaluation plan; the existing test is no longer untouched.

In [ ]:
holdout_predictions=project_search.predict(X_holdout)
print('Final test accuracy:',accuracy_score(y_holdout,holdout_predictions))
print('Final test balanced accuracy:',balanced_accuracy_score(y_holdout,holdout_predictions))
print(classification_report(y_holdout,holdout_predictions,target_names=project_names,zero_division=0))
fig,ax=plt.subplots(figsize=(8,7))
ConfusionMatrixDisplay.from_predictions(y_holdout,holdout_predictions,display_labels=project_names,cmap='Blues',ax=ax,colorbar=False)
ax.set_title('Final held-out test confusion matrix');plt.xticks(rotation=45,ha='right');plt.tight_layout();plt.show()

### Implement G · Manual five-fold selection
Implement the inner grid search without calling GridSearchCV or cross_val_score. Use ParameterGrid, clone the pipeline for every fold, fit on training indices and score validation indices. Return (best parameter dictionary, list of candidate mean scores). Keep the first candidate in a tie. This is an independent implementation of the selection protocol.

In [ ]:
def manual_grid(estimator,grid,X,y,splitter):
    # TODO: iterate candidates and fold splits; fit fresh clones and average validation scores.
    return None

In [ ]:
# A small two-candidate audit; the main project above uses the full stated grid.
audit_grid={'svc__kernel':['linear'],'svc__C':[.1,1]}
audit=manual_grid(project_pipeline,audit_grid,X_develop,y_develop,project_cv)
if audit is None: print('Complete Implement G for the independent selection challenge.')
else:
    reference=GridSearchCV(project_pipeline,audit_grid,cv=project_cv,scoring='accuracy',n_jobs=1,error_score='raise').fit(X_develop,y_develop)
    best_parameters,mean_scores=audit
    np.testing.assert_allclose(mean_scores,reference.cv_results_['mean_test_score'],atol=1e-12)
    assert best_parameters==reference.best_params_
    print('Manual five-fold grid agrees with GridSearchCV:',best_parameters)

**Your reasoning:** write your prediction, observations and explanation here.

### Submit an evidence-based mini-project report
1. Your pre-experiment hypothesis and evaluation protocol.
2. Tuned parameter names, values, candidate count and five-fold fit count. Account separately for baseline and learning-curve fits.
3. Training versus validation performance, descriptive fold spread, model complexity and fit time. Explain why a small CV difference may be inconclusive.
4. A majority-baseline comparison and per-class error analysis.
5. The frozen choice, final test confusion matrix and limitations.
6. A follow-up experiment with its own fresh evaluation plan.

**Independent challenge:** implement manual five-fold grid search with cloned pipelines, the same folds, the same scoring and development-only model selection. Compare with GridSearchCV, documenting tie-breaking, order and seeds. An instructor reference is supplied separately. An optional further project is nested CV: explain the separate roles of inner selection folds and outer evaluation folds.

Your report is evaluated for reasoning and protocol, not for obtaining the highest test accuracy.

## Lecture understanding quiz (15–20 min)

Answer before discussing with classmates. For each answer, add a one-sentence justification and point to the relevant lecture concept.

**Q1. For a correctly labeled point with y*f(x)=0.3, what is its minimum slack?**  
A: 0; B: .3; C: .7; D: 1.3

**Q2. Which points allow direct recovery of b using the zero-slack margin formula?**  
A: all alpha=C; B: all alpha=0; C: 0<alpha<C; D: every point

**Q3. Does alpha=C prove that the point is misclassified?**  
A: yes; B: no

**Q4. 1000 bounded support vectors among 10000 points: possible strict 0/1 training-error range?**  
A: 0 to .1; B: .1 to .1126; C: .1126 to .5; D: .1126 to 1

**Q5. Where should StandardScaler be fitted during cross-validation?**  
Short answer

**Q6. What should you do after selecting C and gamma by development CV?**  
Short answer

**Q7. How many candidates and fold fits come from 4 C values by 3 gamma values with five-fold CV?**  
Short answer

**Q8. Should the best model be selected using the final test confusion matrix?**  
A: yes; B: no

**Q9. If a grid tunes polynomial degree and coef0 as well as C and gamma, how many numerical hyperparameter names are varied?**  
Short answer

**Q10. Is the spread of five CV fold scores automatically a confidence interval?**  
A: yes; B: no

In [ ]:
# Record your answers; this cell does not grade or reveal the key.
quiz_answers = {'Q1': '', 'Q2': '', 'Q3': '', 'Q4': '', 'Q5': '', 'Q6': '', 'Q7': '', 'Q8': '', 'Q9': '', 'Q10': ''}
quiz_justifications = {}
display(quiz_answers)

## Exit ticket
In 3–5 sentences: what did your experiment show, how does the equation explain it, and what question remains? Save the notebook with your completed code, plots, quiz responses and explanations.